# Treino combinado do baseline no Google Colab

Este notebook treina **um unico decoder baseline** com imagens normais combinadas das 15 categorias. Categoria, mascara e label nao sao entradas do modelo. A validacao tambem contem somente imagens normais; o conjunto de teste oficial nao e aberto neste notebook.

## De onde vem o codigo e os dados?

O repositorio publico `renatocorreia-rmcm/allnomalous` contem o codigo e a pasta `dataset/`, com `train_split.csv`, `val_split.csv` e as imagens versionadas com Git LFS. A celula de preparacao instala Git LFS, clona o repositorio e baixa os objetos de imagem para o disco temporario do Colab. O treinamento le esses arquivos locais com os DataLoaders do projeto. Nao e necessario enviar o dataset manualmente ao Drive.

O Drive e usado para checkpoints e curvas persistentes. Se a sessao desconectar, abra o notebook novamente, execute as celulas de preparacao/configuracao/modelo e rode novamente a celula de treino: ela retoma de `last_checkpoint.pt`. Cada bloco salva estado apos cada epoca.

## Configuracao inicial

512x512, batch 2 com acumulacao 2 (batch efetivo 4), AdamW `1e-4`, seed 42, ate 30 epocas, early stopping com paciencia 5 usando somente MSE da validacao normal. A GPU gratuita e a memoria podem variar; rode o preflight antes do treino. Se der falta de memoria, altere batch para 1 e acumulacao para 4 **antes de iniciar**; mantenha essa configuracao ao retomar o checkpoint.

Cada chamada de treino executa ate 3 epocas. Execute a celula novamente para continuar em blocos; o checkpoint no Drive permite tambem retomar apos desconexao. Os pesos ImageNet do extrator sao baixados do Torchvision na primeira execucao.

In [ ]:
# Preparar o codigo e baixar dados publicos via Git LFS.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/renatocorreia-rmcm/allnomalous.git"
REPO_DIR = Path("/content/allnomalous")

def run(command, cwd=None, env=None):
    print("$", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)

if not Path("/usr/bin/git-lfs").exists():
    run(["bash", "-lc", "apt-get update -qq && apt-get install -y git-lfs"])
run(["git", "lfs", "install"])

if not REPO_DIR.exists():
    clone_env = os.environ.copy()
    clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
    run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], env=clone_env)
else:
    run(["git", "pull", "--ff-only"], cwd=REPO_DIR)

run(["git", "lfs", "pull"], cwd=REPO_DIR)
DATA_ROOT = REPO_DIR / "dataset"
assert (DATA_ROOT / "train_split.csv").is_file() and (DATA_ROOT / "val_split.csv").is_file()
assert (DATA_ROOT / "bottle/train/good/000.png").is_file()
print("Codigo, splits e imagens Git LFS disponiveis em", REPO_DIR)

In [ ]:
# Montar Drive para que os checkpoints sobrevivam ao runtime temporario.
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/allnomalous_training")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoints e resultados serao salvos em:", DRIVE_DIR)

In [ ]:
# Colab ja fornece PyTorch/Torchvision com suporte CUDA. Instalar somente utilitarios
# para evitar substituir a build CUDA preinstalada por uma build diferente do pip.
run(["python", "-m", "pip", "install", "-q", "numpy", "Pillow"])

import sys
sys.path.insert(0, str(REPO_DIR))

import torch
import torchvision
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Ative uma GPU em Runtime > Change runtime type antes de continuar.")
print("GPU:", torch.cuda.get_device_name(0))
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"GPU memory free/total: {free_bytes / 2**30:.1f}/{total_bytes / 2**30:.1f} GiB")

In [ ]:
# Configuracao registrada no checkpoint. Nao a altere ao retomar um treino.
import random
import numpy as np

CONFIG = {
    "image_size": 512,
    "batch_size": 2,
    "gradient_accumulation_steps": 2,
    "effective_batch_size": 4,
    "learning_rate": 1e-4,
    "decoder_hidden_dim": 128,
    "decoder_num_encoder_layers": 2,
    "decoder_num_decoder_layers": 2,
    "weight_decay": 1e-4,
    "epochs_max": 30,
    "checkpoint_every_epochs": 1,
    "early_stopping_patience": 5,
    "seed": 42,
    "num_workers": 2,
    "mixed_precision": True,
    "model": "WideResNet50_2_IMAGENET1K_V2_layer1-layer3 + baseline ReconstructionDecoder",
    "jitter": False,
    "neighbor_mask": False,
}
BLOCK_EPOCHS = 3
RESUME = True

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda")

print(CONFIG)
print(f"Per-block limit: {BLOCK_EPOCHS} epochs; maximum total: {CONFIG['epochs_max']}")

In [ ]:
# Criar loaders combinados e confirmar que os splits contem 15 categorias.
from torch.utils.data import DataLoader
from src.mvtec_dataset import MVTecTrainDataset, MVTecValidationDataset

train_dataset = MVTecTrainDataset(DATA_ROOT, image_size=CONFIG["image_size"])
val_dataset = MVTecValidationDataset(DATA_ROOT, image_size=CONFIG["image_size"])
train_generator = torch.Generator().manual_seed(CONFIG["seed"])
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    generator=train_generator,
    persistent_workers=CONFIG["num_workers"] > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    persistent_workers=CONFIG["num_workers"] > 0,
)

train_categories = sorted({path.parts[-4] for path in train_dataset.image_paths})
val_categories = sorted({path.parts[-4] for path in val_dataset.image_paths})
assert len(train_categories) == 15 and len(val_categories) == 15
print(f"Train: {len(train_dataset)} normal images / {len(train_categories)} mixed categories")
print(f"Validation: {len(val_dataset)} normal images / {len(val_categories)} mixed categories")
print("The datasets return images only; category is not passed to the model.")
print("Official test data is not instantiated or read in this notebook.")

## Busca curta de hiperparametros (somente validacao normal)

A busca compara tres configuracoes por duas epocas, usando no maximo 128 imagens normais de treino e 64 de validacao, todas em 512 x 512. Ela varia taxa de aprendizado, largura e profundidade do decoder. O CSV e a figura de mapas normais ficam no Drive; em reinicios, os resultados existentes sao reutilizados. Para refazer a busca, remova esses dois arquivos do Drive. Inspecione os mapas antes de escolher uma configuracao na celula seguinte.

In [ ]:
import csv
import math
import os
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from src.anomaly_maps import compute_anomaly_maps, normalize_map_for_visualization
from src.feature_extractor import WideResNetFeatureExtractor
from src.mvtec_dataset import IMAGENET_MEAN, IMAGENET_STD
from src.reconstruction_decoder import ReconstructionDecoder, reconstruction_loss

TUNING_CANDIDATES = [
    {"name": "baseline", "learning_rate": 1e-4, "hidden_dim": 128, "encoder_layers": 2, "decoder_layers": 2},
    {"name": "higher_lr", "learning_rate": 3e-4, "hidden_dim": 128, "encoder_layers": 2, "decoder_layers": 2},
    {"name": "compact", "learning_rate": 1e-4, "hidden_dim": 64, "encoder_layers": 1, "decoder_layers": 1},
]
TUNING_TRAIN_SAMPLES = 128
TUNING_VALIDATION_SAMPLES = 64
TUNING_EPOCHS = 2
TUNING_CSV_PATH = DRIVE_DIR / "hyperparameter_search.csv"
TUNING_MAPS_PATH = DRIVE_DIR / "hyperparameter_normal_maps.png"

extractor = WideResNetFeatureExtractor(pretrained=True).to(DEVICE).eval()
assert all(not parameter.requires_grad for parameter in extractor.parameters())

if TUNING_CSV_PATH.is_file() and TUNING_MAPS_PATH.is_file():
    with TUNING_CSV_PATH.open(newline="", encoding="utf-8") as handle:
        candidate_results = list(csv.DictReader(handle))
    for result in candidate_results:
        for key in ("learning_rate", "best_val_mse", "normal_mean_top1pct", "normal_mean_map"):
            result[key] = float(result[key])
        for key in ("hidden_dim", "encoder_layers", "decoder_layers", "best_epoch"):
            result[key] = int(result[key])
    print("Loaded prior validation-only tuning results from Drive.")
    display(plt.imread(TUNING_MAPS_PATH))
else:
    train_generator = torch.Generator().manual_seed(CONFIG["seed"])
    val_generator = torch.Generator().manual_seed(CONFIG["seed"] + 1)
    train_indices = torch.randperm(len(train_dataset), generator=train_generator)[:TUNING_TRAIN_SAMPLES].tolist()
    val_indices = torch.randperm(len(val_dataset), generator=val_generator)[:TUNING_VALIDATION_SAMPLES].tolist()
    tuning_train_loader = DataLoader(
        Subset(train_dataset, train_indices), batch_size=CONFIG["batch_size"],
        shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(CONFIG["seed"]),
    )
    tuning_val_loader = DataLoader(
        Subset(val_dataset, val_indices), batch_size=CONFIG["batch_size"],
        shuffle=False, num_workers=0,
    )
    visual_images = torch.stack([val_dataset[index] for index in val_indices[:4]])
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    candidate_results = []
    candidate_visual_maps = {}

    for candidate_index, candidate in enumerate(TUNING_CANDIDATES):
        torch.manual_seed(CONFIG["seed"] + candidate_index)
        torch.cuda.manual_seed_all(CONFIG["seed"] + candidate_index)
        tuning_decoder = ReconstructionDecoder(
            in_channels=1792,
            hidden_dim=candidate["hidden_dim"],
            num_heads=4,
            num_encoder_layers=candidate["encoder_layers"],
            num_decoder_layers=candidate["decoder_layers"],
            feedforward_dim=candidate["hidden_dim"] * 4,
            dropout=0.1,
            base_grid_size=(32, 32),
        ).to(DEVICE)
        tuning_optimizer = torch.optim.AdamW(
            tuning_decoder.parameters(), lr=candidate["learning_rate"], weight_decay=CONFIG["weight_decay"]
        )
        tuning_scaler = torch.cuda.amp.GradScaler(enabled=CONFIG["mixed_precision"])
        best_val_mse = float("inf")
        best_epoch = 0
        best_state = None

        for epoch in range(1, TUNING_EPOCHS + 1):
            tuning_decoder.train()
            for images in tuning_train_loader:
                images = images.to(DEVICE, non_blocking=True)
                with torch.no_grad(), torch.autocast("cuda", enabled=CONFIG["mixed_precision"]):
                    targets = extractor(images)
                with torch.autocast("cuda", enabled=CONFIG["mixed_precision"]):
                    reconstruction = tuning_decoder(targets)
                    loss = reconstruction_loss(targets, reconstruction)
                if not torch.isfinite(loss):
                    raise RuntimeError(f"Non-finite tuning loss for candidate {candidate['name']}")
                tuning_optimizer.zero_grad(set_to_none=True)
                tuning_scaler.scale(loss).backward()
                tuning_scaler.step(tuning_optimizer)
                tuning_scaler.update()

            tuning_decoder.eval()
            val_loss_sum = 0.0
            val_sample_count = 0
            with torch.no_grad():
                for images in tuning_val_loader:
                    images = images.to(DEVICE, non_blocking=True)
                    with torch.autocast("cuda", enabled=CONFIG["mixed_precision"]):
                        targets = extractor(images)
                        reconstruction = tuning_decoder(targets)
                        val_loss = reconstruction_loss(targets, reconstruction)
                    val_loss_sum += val_loss.item() * images.shape[0]
                    val_sample_count += images.shape[0]
            epoch_val_mse = val_loss_sum / val_sample_count
            if not math.isfinite(epoch_val_mse):
                raise RuntimeError(f"Non-finite validation loss for candidate {candidate['name']}")
            print(f"{candidate['name']} epoch {epoch}/{TUNING_EPOCHS}: normal val MSE={epoch_val_mse:.6f}")
            if epoch_val_mse < best_val_mse:
                best_val_mse = epoch_val_mse
                best_epoch = epoch
                best_state = {key: value.detach().cpu().clone() for key, value in tuning_decoder.state_dict().items()}

        tuning_decoder.load_state_dict(best_state)
        tuning_decoder.eval()
        normal_scores = []
        normal_map_means = []
        visual_maps = []
        with torch.no_grad():
            for images in tuning_val_loader:
                images = images.to(DEVICE, non_blocking=True)
                with torch.autocast("cuda", enabled=CONFIG["mixed_precision"]):
                    targets = extractor(images)
                    reconstruction = tuning_decoder(targets)
                maps = compute_anomaly_maps(
                    targets, reconstruction, output_size=(CONFIG["image_size"], CONFIG["image_size"]),
                    smoothing_kernel_size=5,
                )
                topk = max(1, int(0.01 * maps.shape[-2] * maps.shape[-1]))
                normal_scores.extend(maps.flatten(1).topk(topk, dim=1).values.mean(1).cpu().tolist())
                normal_map_means.extend(maps.flatten(1).mean(1).cpu().tolist())
                if len(visual_maps) < len(visual_images):
                    remaining = len(visual_images) - len(visual_maps)
                    visual_maps.extend(maps[:remaining].cpu())
        candidate_visual_maps[candidate["name"]] = torch.stack(visual_maps)
        candidate_results.append({
            "name": candidate["name"],
            "learning_rate": candidate["learning_rate"],
            "hidden_dim": candidate["hidden_dim"],
            "encoder_layers": candidate["encoder_layers"],
            "decoder_layers": candidate["decoder_layers"],
            "best_epoch": best_epoch,
            "best_val_mse": best_val_mse,
            "normal_mean_top1pct": sum(normal_scores) / len(normal_scores),
            "normal_mean_map": sum(normal_map_means) / len(normal_map_means),
            "official_test_used": False,
        })
        del tuning_decoder, tuning_optimizer, tuning_scaler, best_state
        torch.cuda.empty_cache()

    fields = ["name", "learning_rate", "hidden_dim", "encoder_layers", "decoder_layers", "best_epoch", "best_val_mse", "normal_mean_top1pct", "normal_mean_map", "official_test_used"]
    temporary_tuning_path = TUNING_CSV_PATH.with_suffix(".csv.tmp")
    with temporary_tuning_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(candidate_results)
    os.replace(temporary_tuning_path, TUNING_CSV_PATH)

    fig, axes = plt.subplots(len(TUNING_CANDIDATES), 8, figsize=(18, 7))
    for row_index, candidate in enumerate(TUNING_CANDIDATES):
        maps = candidate_visual_maps[candidate["name"]]
        for image_index in range(4):
            image = (visual_images[image_index] * std[0] + mean[0]).clamp(0, 1).permute(1, 2, 0).numpy()
            anomaly_map = normalize_map_for_visualization(maps[image_index].unsqueeze(0))[0].numpy()
            axes[row_index, image_index * 2].imshow(image)
            axes[row_index, image_index * 2].set_title("Normal validation image" if row_index == 0 else "")
            axes[row_index, image_index * 2 + 1].imshow(image)
            axes[row_index, image_index * 2 + 1].imshow(anomaly_map, cmap="jet", alpha=0.45, vmin=0, vmax=1)
            axes[row_index, image_index * 2 + 1].set_title("Raw map (display-normalized)" if row_index == 0 else "")
            axes[row_index, image_index * 2].set_ylabel(candidate["name"])
            axes[row_index, image_index * 2].axis("off")
            axes[row_index, image_index * 2 + 1].axis("off")
    fig.tight_layout()
    fig.savefig(TUNING_MAPS_PATH, dpi=150)
    display(fig)
    plt.close(fig)

print("Validation-only results (lower MSE and normal-map scores are preferable; visually inspect the maps):")
for result in sorted(candidate_results, key=lambda row: row["best_val_mse"]):
    print(result)
print("Results CSV:", TUNING_CSV_PATH)
print("Normal-validation maps:", TUNING_MAPS_PATH)

Escolha uma linha da tabela depois de inspecionar a figura. O menor MSE e a menor intensidade top-1% em validacao normal ajudam a comparar, mas nao substituem a inspecao visual. Preencha o nome exato e a justificativa antes de continuar; nenhum split de teste e usado.

In [ ]:
SELECTED_CANDIDATE = ""  # baseline, higher_lr ou compact
SELECTION_JUSTIFICATION = ""  # cite a validacao normal e a inspecao visual dos mapas

if not SELECTED_CANDIDATE or not SELECTION_JUSTIFICATION.strip():
    raise RuntimeError("Inspect the validation table/maps, then fill SELECTED_CANDIDATE and SELECTION_JUSTIFICATION.")
selected = next((row for row in candidate_results if row["name"] == SELECTED_CANDIDATE), None)
if selected is None:
    raise ValueError(f"Unknown candidate {SELECTED_CANDIDATE!r}; choose one of {[row['name'] for row in candidate_results]}.")
for result in candidate_results:
    result["selected"] = result["name"] == SELECTED_CANDIDATE
    result["selection_justification"] = SELECTION_JUSTIFICATION.strip() if result["selected"] else ""
tuning_fields = ["name", "learning_rate", "hidden_dim", "encoder_layers", "decoder_layers", "best_epoch", "best_val_mse", "normal_mean_top1pct", "normal_mean_map", "official_test_used", "selected", "selection_justification"]
temporary_tuning_path = TUNING_CSV_PATH.with_suffix(".csv.tmp")
with temporary_tuning_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=tuning_fields)
    writer.writeheader()
    writer.writerows(candidate_results)
os.replace(temporary_tuning_path, TUNING_CSV_PATH)
CONFIG.update({
    "learning_rate": float(selected["learning_rate"]),
    "decoder_hidden_dim": int(selected["hidden_dim"]),
    "decoder_num_encoder_layers": int(selected["encoder_layers"]),
    "decoder_num_decoder_layers": int(selected["decoder_layers"]),
    "selected_candidate": SELECTED_CANDIDATE,
    "selection_justification": SELECTION_JUSTIFICATION.strip(),
})
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])
print("Selected final config:", {key: CONFIG[key] for key in ["selected_candidate", "learning_rate", "decoder_hidden_dim", "decoder_num_encoder_layers", "decoder_num_decoder_layers", "selection_justification"]})

In [ ]:
# Inicializar extrator e decoder baseline. O extrator fica congelado.
from src.reconstruction_decoder import ReconstructionDecoder, reconstruction_loss

decoder = ReconstructionDecoder(
    in_channels=1792,
    hidden_dim=CONFIG["decoder_hidden_dim"],
    num_heads=4,
    num_encoder_layers=CONFIG["decoder_num_encoder_layers"],
    num_decoder_layers=CONFIG["decoder_num_decoder_layers"],
    feedforward_dim=CONFIG["decoder_hidden_dim"] * 4,
    dropout=0.1,
    base_grid_size=(32, 32),
    neighbor_size=None,
    jitter_scale=0.0,
    jitter_probability=0.0,
).to(DEVICE)
optimizer = torch.optim.AdamW(
    decoder.parameters(),
    lr=CONFIG["learning_rate"],
    betas=(0.9, 0.999),
    weight_decay=CONFIG["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG["epochs_max"]
)
scaler = torch.cuda.amp.GradScaler(enabled=CONFIG["mixed_precision"])
total_parameters = sum(parameter.numel() for parameter in extractor.parameters()) + sum(
    parameter.numel() for parameter in decoder.parameters()
)
trainable_parameters = sum(
    parameter.numel() for parameter in list(extractor.parameters()) + list(decoder.parameters())
    if parameter.requires_grad
)
print(f"Model parameters: {total_parameters:,} total / {trainable_parameters:,} trainable")
print("Frozen extractor and baseline decoder initialized.")

In [ ]:
# GPU preflight: one training batch, forward/backward, peak memory and latency.
import time

preflight_cpu_rng = torch.get_rng_state()
preflight_cuda_rng = torch.cuda.get_rng_state_all()
preflight_loader_rng = train_generator.get_state()
preflight_images = next(iter(train_loader)).to(DEVICE, non_blocking=True)
preflight_decoder = ReconstructionDecoder(
    in_channels=1792, hidden_dim=CONFIG["decoder_hidden_dim"], num_heads=4,
    num_encoder_layers=CONFIG["decoder_num_encoder_layers"],
    num_decoder_layers=CONFIG["decoder_num_decoder_layers"],
    feedforward_dim=CONFIG["decoder_hidden_dim"] * 4,
    dropout=0.1, base_grid_size=(32, 32),
).to(DEVICE).train()
torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
with torch.no_grad(), torch.autocast("cuda", enabled=CONFIG["mixed_precision"]):
    preflight_features = extractor(preflight_images)
with torch.autocast("cuda", enabled=CONFIG["mixed_precision"]):
    preflight_reconstruction = preflight_decoder(preflight_features)
    preflight_loss = reconstruction_loss(preflight_features, preflight_reconstruction)
preflight_loss.backward()
torch.cuda.synchronize()
print("Batch:", tuple(preflight_images.shape))
print("Features/reconstruction:", tuple(preflight_features.shape), tuple(preflight_reconstruction.shape))
print("Finite loss:", torch.isfinite(preflight_loss).item(), "MSE:", preflight_loss.item())
print(f"One-batch forward/backward: {time.perf_counter() - start:.1f} s")
print(f"Peak allocated: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
del preflight_images, preflight_decoder, preflight_features, preflight_reconstruction, preflight_loss
torch.set_rng_state(preflight_cpu_rng)
torch.cuda.set_rng_state_all([state.cpu() for state in preflight_cuda_rng])
train_generator.set_state(preflight_loader_rng)
torch.cuda.empty_cache()
print("If this cell runs out of memory, restart with batch_size=1 and accumulation=4 before training.")

In [ ]:
# Checkpoints retomaveis no Drive e treino em blocos.
import csv
import json
import math
import os
import time
import matplotlib.pyplot as plt

LAST_PATH = DRIVE_DIR / "last_checkpoint.pt"
BEST_PATH = DRIVE_DIR / "best_decoder.pt"
HISTORY_PATH = DRIVE_DIR / "loss_history.csv"
PLOT_PATH = DRIVE_DIR / "loss_curves.png"
META_PATH = DRIVE_DIR / "training_config.json"
META_PATH.write_text(json.dumps({
    "config": CONFIG,
    "repo_url": REPO_URL,
    "repo_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "total_parameters": total_parameters,
    "trainable_parameters": trainable_parameters,
}, indent=2), encoding="utf-8")

history = []
start_epoch = 0
best_val_loss = math.inf
epochs_without_improvement = 0
total_train_seconds = 0.0

def _atomic_torch_save(payload, path):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary_path)
    os.replace(temporary_path, path)

def _save_history_and_plot():
    if not history:
        return
    fields = ["epoch", "train_loss", "val_loss", "epoch_seconds", "learning_rate"]
    temporary_csv = HISTORY_PATH.with_suffix(".csv.tmp")
    with temporary_csv.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(history)
    os.replace(temporary_csv, HISTORY_PATH)

    epochs = [item["epoch"] for item in history]
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, [item["train_loss"] for item in history], marker="o", label="Train")
    plt.plot(epochs, [item["val_loss"] for item in history], marker="o", label="Normal validation")
    plt.xlabel("Epoch")
    plt.ylabel("Reconstruction MSE")
    plt.title("Combined 15-category baseline")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOT_PATH, dpi=160)
    plt.show()
    plt.close()

def save_epoch_checkpoint(epoch):
    global history, best_val_loss, epochs_without_improvement, total_train_seconds
    payload = {
        "epoch": epoch,
        "config": CONFIG,
        "decoder_state_dict": decoder.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_loss": best_val_loss,
        "epochs_without_improvement": epochs_without_improvement,
        "history": history,
        "total_train_seconds": total_train_seconds,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_states": torch.cuda.get_rng_state_all(),
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "loader_generator_state": train_generator.get_state(),
    }
    _atomic_torch_save(payload, LAST_PATH)
    _save_history_and_plot()

def load_latest_checkpoint():
    global history, start_epoch, best_val_loss, epochs_without_improvement, total_train_seconds
    if not RESUME or not LAST_PATH.exists():
        print("Starting a new run.")
        return
    try:
        checkpoint = torch.load(LAST_PATH, map_location=DEVICE, weights_only=False)
    except TypeError:
        checkpoint = torch.load(LAST_PATH, map_location=DEVICE)
    if checkpoint["config"] != CONFIG:
        raise ValueError("Saved checkpoint config differs from CONFIG. Restore the original settings before resuming.")
    decoder.load_state_dict(checkpoint["decoder_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    if checkpoint.get("scaler_state_dict"):
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch = checkpoint["epoch"]
    best_val_loss = checkpoint["best_val_loss"]
    epochs_without_improvement = checkpoint["epochs_without_improvement"]
    history = checkpoint["history"]
    total_train_seconds = checkpoint["total_train_seconds"]
    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    torch.cuda.set_rng_state_all([state.cpu() for state in checkpoint["cuda_rng_states"]])
    random.setstate(checkpoint["python_rng_state"])
    np.random.set_state(checkpoint["numpy_rng_state"])
    train_generator.set_state(checkpoint["loader_generator_state"].cpu())
    print(f"Resumed after epoch {start_epoch}; best validation MSE={best_val_loss:.6f}")

def run_epoch(epoch):
    decoder.train()
    total_loss = 0.0
    seen = 0
    optimizer.zero_grad(set_to_none=True)
    total_batches = len(train_loader)
    epoch_started = time.perf_counter()

    for batch_index, images in enumerate(train_loader):
        images = images.to(DEVICE, non_blocking=True)
        with torch.no_grad(), torch.autocast("cuda", enabled=CONFIG["mixed_precision"]):
            targets = extractor(images)
        with torch.autocast("cuda", enabled=CONFIG["mixed_precision"]):
            reconstruction = decoder(targets)
            loss = reconstruction_loss(targets, reconstruction)
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite training loss at epoch {epoch}")

        group_start = (batch_index // CONFIG["gradient_accumulation_steps"]) * CONFIG["gradient_accumulation_steps"]
        group_size = min(CONFIG["gradient_accumulation_steps"], total_batches - group_start)
        scaler.scale(loss / group_size).backward()
        batch_count = images.shape[0]
        total_loss += loss.item() * batch_count
        seen += batch_count

        step_now = (
            (batch_index + 1) % CONFIG["gradient_accumulation_steps"] == 0
            or batch_index + 1 == total_batches
        )
        if step_now:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

    train_loss = total_loss / seen
    decoder.eval()
    validation_total = 0.0
    validation_seen = 0
    with torch.no_grad():
        for images in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", enabled=CONFIG["mixed_precision"]):
                targets = extractor(images)
                reconstruction = decoder(targets)
                loss = reconstruction_loss(targets, reconstruction)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite validation loss at epoch {epoch}")
            batch_count = images.shape[0]
            validation_total += loss.item() * batch_count
            validation_seen += batch_count

    val_loss = validation_total / validation_seen
    scheduler.step()
    epoch_seconds = time.perf_counter() - epoch_started
    return train_loss, val_loss, epoch_seconds

def train_block(epochs_this_block=BLOCK_EPOCHS):
    global start_epoch, best_val_loss, epochs_without_improvement, total_train_seconds
    if epochs_this_block < 1:
        raise ValueError("epochs_this_block must be positive")
    block_end = min(CONFIG["epochs_max"], start_epoch + epochs_this_block)
    while start_epoch < block_end and epochs_without_improvement < CONFIG["early_stopping_patience"]:
        epoch = start_epoch + 1
        train_loss, val_loss, epoch_seconds = run_epoch(epoch)
        total_train_seconds += epoch_seconds
        current_lr = optimizer.param_groups[0]["lr"]
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "epoch_seconds": epoch_seconds,
            "learning_rate": current_lr,
        })
        start_epoch = epoch

        improved = val_loss < best_val_loss
        if improved:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            _atomic_torch_save({
                "epoch": epoch,
                "config": CONFIG,
                "validation_loss": best_val_loss,
                "decoder_state_dict": decoder.state_dict(),
            }, BEST_PATH)
        else:
            epochs_without_improvement += 1

        should_checkpoint = (
            epoch % CONFIG["checkpoint_every_epochs"] == 0
            or epoch == block_end
            or epoch == CONFIG["epochs_max"]
            or epochs_without_improvement >= CONFIG["early_stopping_patience"]
        )
        if should_checkpoint:
            save_epoch_checkpoint(epoch)
        else:
            _save_history_and_plot()
        print(
            f"epoch {epoch:02d}/{CONFIG['epochs_max']} | "
            f"train MSE={train_loss:.6f} | val MSE={val_loss:.6f} | "
            f"{epoch_seconds / 60:.1f} min | best val={best_val_loss:.6f}"
        )
        if epochs_without_improvement >= CONFIG["early_stopping_patience"]:
            print("Early stopping: validation normal loss did not improve.")
            break

    if start_epoch >= CONFIG["epochs_max"]:
        print("Maximum epoch count reached.")
    print(f"Latest resumable checkpoint: {LAST_PATH}")
    print(f"Best decoder checkpoint: {BEST_PATH}")
    print(f"Loss plot: {PLOT_PATH}")
    print(f"Accumulated training/validation time: {total_train_seconds / 3600:.2f} hours")

load_latest_checkpoint()
print(f"Ready. Run train_block() in blocks of {BLOCK_EPOCHS}; checkpoint interval: {CONFIG['checkpoint_every_epochs']} epoch(s).")

## Testar retomada antes do treino completo

Para verificar o ciclo de retomada, execute `train_block(epochs_this_block=1)` uma vez. Depois use **Runtime > Restart session** e execute novamente as celulas desde o clone ate `load_latest_checkpoint()`; a busca reaproveita os resultados salvos no Drive. Confirme a mensagem `Resumed after epoch 1`, execute mais uma vez `train_block(epochs_this_block=1)` e confirme que o treino avanca para a epoca 2 e atualiza `last_checkpoint.pt` e `loss_history.csv`. Esse teste treina uma epoca real; os resultados podem ser mantidos como parte do treino.

In [ ]:
# Execute one block. Rerun this cell to continue; each completed epoch is saved to Drive.
train_block()

In [ ]:
# Inspect the saved training curve and checkpoint locations.
if PLOT_PATH.exists():
    display(plt.imread(PLOT_PATH))
print("Latest checkpoint:", LAST_PATH)
print("Best model:", BEST_PATH)
print("Loss history:", HISTORY_PATH)
print("Total training time (hours):", round(total_train_seconds / 3600, 3))
print("To download/share the checkpoint, open Google Drive > My Drive > allnomalous_training.")

In [ ]:
# Run after training (and again after the final block) to export the cost table.
import csv
import time

if not BEST_PATH.is_file() or not LAST_PATH.is_file() or not history:
    raise RuntimeError("Train at least one epoch before measuring model and checkpoint costs.")

print("Benchmarking the latest resumable decoder checkpoint.")
decoder.eval()
extractor.eval()

cost_rows = [
    {"metric": "gpu", "value": torch.cuda.get_device_name(0), "unit": "model", "details": "Colab runtime GPU"},
    {"metric": "image_size", "value": CONFIG["image_size"], "unit": "pixels", "details": "square input"},
    {"metric": "total_parameters", "value": total_parameters, "unit": "parameters", "details": "frozen extractor + decoder"},
    {"metric": "trainable_parameters", "value": trainable_parameters, "unit": "parameters", "details": "decoder only"},
    {"metric": "completed_epochs", "value": start_epoch, "unit": "epochs", "details": "resumed training history"},
    {"metric": "total_epoch_time", "value": round(total_train_seconds, 3), "unit": "seconds", "details": "training + validation; excludes checkpoint I/O"},
    {"metric": "average_epoch_time", "value": round(sum(item["epoch_seconds"] for item in history) / len(history), 3), "unit": "seconds", "details": "training + validation"},
    {"metric": "latest_checkpoint_size", "value": round(LAST_PATH.stat().st_size / 1_000_000, 3), "unit": "MB", "details": LAST_PATH.name},
    {"metric": "best_checkpoint_size", "value": round(BEST_PATH.stat().st_size / 1_000_000, 3), "unit": "MB", "details": BEST_PATH.name},
]

sample_images = next(iter(val_loader))[:1].to(DEVICE, non_blocking=True)
def inference_forward(images):
    with torch.inference_mode(), torch.autocast("cuda", enabled=CONFIG["mixed_precision"]):
        features = extractor(images)
        return decoder(features)

WARMUP_ITERATIONS = 10
MEASURED_ITERATIONS = 50
# If batch 4 does not fit on the selected GPU, retry with (1, 2).
INFERENCE_BATCH_SIZES = (1, 4)
for batch_size in INFERENCE_BATCH_SIZES:
    inference_images = sample_images.expand(batch_size, -1, -1, -1).contiguous()
    for _ in range(WARMUP_ITERATIONS):
        inference_output = inference_forward(inference_images)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    for _ in range(MEASURED_ITERATIONS):
        inference_output = inference_forward(inference_images)
    torch.cuda.synchronize()
    batch_latency_ms = (time.perf_counter() - started) * 1000 / MEASURED_ITERATIONS
    peak_memory_gb = torch.cuda.max_memory_allocated() / 2**30
    cost_rows.extend([
        {"metric": f"inference_batch_{batch_size}_latency", "value": round(batch_latency_ms, 3), "unit": "ms/batch", "details": f"full extractor + decoder; {WARMUP_ITERATIONS} warmups, {MEASURED_ITERATIONS} runs"},
        {"metric": f"inference_batch_{batch_size}_latency_per_image", "value": round(batch_latency_ms / batch_size, 3), "unit": "ms/image", "details": "batch latency divided by batch size"},
        {"metric": f"inference_batch_{batch_size}_peak_memory", "value": round(peak_memory_gb, 3), "unit": "GiB", "details": "peak allocated GPU memory"},
    ])
    del inference_images, inference_output

COSTS_PATH = DRIVE_DIR / "computational_costs.csv"
temporary_costs_path = COSTS_PATH.with_suffix(".csv.tmp")
with temporary_costs_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["metric", "value", "unit", "details"])
    writer.writeheader()
    writer.writerows(cost_rows)
os.replace(temporary_costs_path, COSTS_PATH)
print("Cost table:", COSTS_PATH)
for row in cost_rows:
    print(f"{row['metric']}: {row['value']} {row['unit']} ({row['details']})")